In [1]:
import os
import polars as pl

experiment = "Final"
model_types = ["rf", "lgb",  "dt", "lr",'mlp']

# Metriche comuni a tutti i modelli
metrics = ["AUC", "F1", "precision", "recall", "accuracy", "accuracy_train" ] #, "F1_train","average_precision"

# Iperparametri specifici per ogni modello
hyperparams = {
    "rf":  ["rf_n_estimators", "rf_max_depth", "rf_min_samples_leaf"],

    "lgb": ["lgb_n_estimators", "lgb_max_depth", "learning_rate", "min_child_weight",
             "subsample", "colsample_bytree", "reg_alpha", "reg_lambda"],

    "mlp": ["hidden_sizes", "mlp_learning_rate", "dropout_rate",
             "weight_decay", "batch_size"],

    "dt":  ["dt_max_depth", "dt_criterion", "dt_min_samples_split", "dt_min_samples_leaf"],

    "lr":  ["lr_C", "lr_penalty"],
}

# Metrica di ordinamento
sort_metric = "F1"

results_dir = f"data/results/{experiment}"
output_path = f"data/results/{experiment}/summary.txt"
results = {}
lines = []

for model in model_types:
    path = os.path.join(results_dir, f"{model}.csv")
    df = pl.read_csv(path)

    # Solo run completate con metrica valida
    df = df.filter(
        (pl.col("State") == "finished") & pl.col(sort_metric).is_not_null()
    )

    cols = hyperparams[model] + metrics
    df_selected = df.select(["Name"] + cols).sort(sort_metric, descending=True)

    # Cast a float e arrotondamento metriche a 2 decimali
    df_selected = df_selected.with_columns(
        [pl.col(m).cast(pl.Float64, strict=False).round(2) for m in metrics]
    )

    results[model] = df_selected
    top = df_selected.row(0, named=True)

    header = f"\n{'='*40}\n {model.upper()} — best by {sort_metric} ({top['Name']})\n{'='*40}"
    print(header)
    lines.append(header)

    print("\n  Hyperparameters:")
    lines.append("\n  Hyperparameters:")
    for h in hyperparams[model]:
        line = f"    {h:25s} {top[h]}"
        print(line)
        lines.append(line)

    print("\n  Metrics:")
    lines.append("\n  Metrics:")
    for m in metrics:
        line = f"    {m:25s} {top[m]}"
        print(line)
        lines.append(line)

with open(output_path, "w") as f:
    f.write("\n".join(lines))

print(f"\n\nSaved to {output_path}")


 RF — best by F1 (quiet-sweep-66)

  Hyperparameters:
    rf_n_estimators           300
    rf_max_depth              7
    rf_min_samples_leaf       9

  Metrics:
    AUC                       0.81
    F1                        0.64
    precision                 0.59
    recall                    0.7
    accuracy                  0.75
    accuracy_train            0.75

 LGB — best by F1 (azure-sweep-13)

  Hyperparameters:
    lgb_n_estimators          200
    lgb_max_depth             15
    learning_rate             0.024772795314626415
    min_child_weight          10
    subsample                 0.8
    colsample_bytree          0.6
    reg_alpha                 0.19405237273916945
    reg_lambda                4.782083257752451

  Metrics:
    AUC                       0.82
    F1                        0.65
    precision                 0.58
    recall                    0.72
    accuracy                  0.75
    accuracy_train            0.77

 DT — best by F1 (vivid-sweep-

In [7]:
# --- Compare with another experiment's all.csv ---
other_label = "No-window"  
other_path = f"data/results/{other_label}/all.csv"  # <-- change this path as needed
current_label = experiment

# Read the other experiment's results and pick best F1 per model
df_other = pl.read_csv(other_path)
df_other = df_other.filter(
    (pl.col("State") == "finished") & pl.col(sort_metric).is_not_null()
)
best_other = (
    df_other
    .group_by("model_type")
    .agg([pl.col(m).cast(pl.Float64, strict=False).max().round(2).alias(m) for m in metrics])
    .sort("model_type")
)

# Build best-per-model from current experiment
rows = []
for model, df_selected in results.items():
    top = df_selected.row(0, named=True)
    row = {"model_type": model}
    for m in metrics:
        row[m] = top[m]
    rows.append(row)
best_current = pl.DataFrame(rows).sort("model_type")

# Compare side by side
print(f"{'Model':<6} {'Metric':<20} {current_label:>15} {other_label:>15} {'Diff':>10} {'Diff%':>10}")
print("-" * 80)

for model in sorted(set(best_current["model_type"]) & set(best_other["model_type"])):
    cur_row = best_current.filter(pl.col("model_type") == model).row(0, named=True)
    oth_row = best_other.filter(pl.col("model_type") == model).row(0, named=True)
    for m in metrics:
        cur_val = cur_row[m] if cur_row[m] is not None else float("nan")
        oth_val = oth_row[m] if oth_row[m] is not None else float("nan")
        valid = cur_val == cur_val and oth_val == oth_val
        diff = round(oth_val -cur_val, 2) if valid else float("nan")
        pct = round((oth_val - cur_val ) / cur_val * 100, 1) if valid and cur_val != 0 else float("nan")
        sign_d = "+" if diff > 0 else ""
        sign_p = "+" if pct > 0 else ""
        pct_str = f"{sign_p}{pct}%" if pct == pct else "nan"
        print(f"{model:<6} {m:<20} {cur_val:>15} {oth_val:>15} {sign_d + str(diff):>10} {pct_str:>10}")
    print()


Model  Metric                         Final       No-window       Diff      Diff%
--------------------------------------------------------------------------------
dt     AUC                             0.79            0.84      +0.05      +6.3%
dt     F1                              0.63            0.74      +0.11     +17.5%
dt     precision                       0.57            0.69      +0.12     +21.1%
dt     recall                           0.7             0.8       +0.1     +14.3%
dt     accuracy                        0.73            0.77      +0.04      +5.5%
dt     accuracy_train                  0.74            0.78      +0.04      +5.4%

lgb    AUC                             0.82            0.88      +0.06      +7.3%
lgb    F1                              0.65            0.77      +0.12     +18.5%
lgb    precision                       0.58            0.73      +0.15     +25.9%
lgb    recall                          0.72            0.81      +0.09     +12.5%
lgb    accuracy 